# prompt

> Run a prompt in the current kernel.

In [ ]:
#| default_exp prompt

In [ ]:
import os, json, sys, asyncio, base64
from contextlib import aclosing
from datetime import datetime
from pathlib import Path
from inspect import isawaitable
from IPython import get_ipython
from IPython.display import display, Markdown
from IPython.utils.capture import capture_output
from fastcore.foundation import L
from fastcore.xtras import obj2dict, dict2obj
from fastcore.nbio import dict2nb, repair_nb
from fastcore.xml import ft, to_xml, Variable_Errors, Tool_Errors, Context_Warning, Safe
from fastllm.chat import AsyncChat, StreamAccum
from aidialog.msg_parts import mk_msgs, ToolResponse, StopResponse, MdStr
from fasttransport.core import AsyncTransport
from aidialog.dialog import Message, sprompt, prompt_output, get_output_mds, INTERRUPTED, scode
from aidialog.hist import chat2dlg, get_exprs, get_refs, is_nameerr, vars_hist, warning_tag, output_parts, merge_media
import aidialog.ipynb
from dialoghelper.core import cells_client, _add_msg_unsafe

`run_prompt` sends a prompt to Solve-LP's `/dlg_responses` endpoint. A stored cell ID includes preceding notebook history. Passing `solveit` settings additionally sends a `dialog` block with inherited CRAFTs, the notebook's settings, and resolved namespace references, which Solve-LP expands into the Solveit system instructions, ICL examples, and warnings. Preparation runs in the kernel when execution starts, while Solve-LP handles skipped/pinned cells and context-window fitting. Passing prompt text directly remains a standalone request. The model's `py` and `spawn_agent` tool calls run in the kernel during the turn. The caller selects the model and reasoning effort; `SOLVELP_URL`, `AAI_USER_KEY`, and `INSTANCE_ID` provide the connection.

Await the function in a code cell. It updates one Markdown display and returns no value. Each update carries `is_ai_res` metadata so Solveit's shared renderer formats usage and tool details just like a server-generated reply. The kernel emits raw Markdown, not HTML, preserving the stored reply for copying and future context. A failed request ends the display with a user-facing error note. Interrupting the execution ends it with the standard interruption note and aborts the queued tail like an interrupted code cell.

In [ ]:
def _prompt_body(cells, id):
    "Select a stored prompt and its preceding history without changing the snapshot."
    for i,cell in enumerate(cells):
        if cell['id'] != id: continue
        if not cell['metadata'].get('solveit_ai'): raise ValueError(f'cell {id} is not a prompt')
        return dict(cells=obj2dict(cells[:i]), prompt=obj2dict(cell) | dict(outputs=[]))
    raise KeyError(f'prompt {id} no longer exists')

For a stored prompt, history is a snapshot taken when its queued execution starts. Only cells above that prompt are sent, with their outputs, attachments, and skipped/pinned metadata intact. Solve-LP applies visibility rules and fits the history to the selected model. The current prompt's old answer is excluded, without changing the snapshot.

In [ ]:
from fastcore.test import *
import operator
from fasttransport.errors import APIError
from fastcore.nbio import mk_cell
from aidialog.dialog import prompt_output
from safepyrun import RunPython

In [ ]:
notes = mk_cell('The mascot is a wombat.', cell_type='markdown', id='notes', metadata=dict(pinned=True))
code = mk_cell('6 * 7', id='code', outputs=[dict(output_type='execute_result', execution_count=1, data={'text/plain':'42'}, metadata={})])
previous = Message('What is the mascot?', msg_type=sprompt, id='previous', output=prompt_output('A wombat.')).to_cell()
skipped = mk_cell('Ignore this note.', cell_type='markdown', id='skipped', metadata=dict(skipped=True))
target = Message('What did we learn?', msg_type=sprompt, id='target', output=prompt_output('An old answer.')).to_cell()
target['attachments'] = {'photo':{'image/png':'aGVsbG8='}}
snapshot = [notes, code, previous, skipped, target, mk_cell('Future note', cell_type='markdown', id='future')]
body = _prompt_body(snapshot, 'target')
test_eq([c['id'] for c in body['cells']], ['notes', 'code', 'previous', 'skipped'])
test_eq(body['cells'], obj2dict(snapshot[:4]))
test_eq(body['prompt']['outputs'], [])
test_eq(body['prompt']['attachments'], target['attachments'])
assert target['outputs']
with expect_fail(KeyError): _prompt_body(snapshot, 'missing')
with expect_fail(ValueError): _prompt_body(snapshot, 'notes')
[c['id'] for c in body['cells']]

In [ ]:
def _craft_messages(files, name):
    msgs,errors = [Message('# Loaded CRAFTs.', msg_type='note', id='_crafts_h1')],[]
    for f in files:
        if f['path']==name: continue
        try:
            nb = json.loads(f['content'])
            repair_nb(nb)
            cells = dict2nb(nb).cells
        except ValueError as e:
            errors.append(f'Could not read {f["path"]}: {e}')
            continue
        cid = f'_craft_{hash(f["path"]) & 0xffffffff:08x}'
        msgs.append(Message(f'<craft-block from-dialog="/{f["path"]}">', msg_type='raw', id=cid))
        msgs.extend(Message.from_cell(c) for c in cells)
        msgs.append(Message('</craft-block>', msg_type='raw', id=cid+'_end'))
    msgs.append(Message('End of CRAFTs.', msg_type=sprompt, output=prompt_output('Ok!'), id='_craft_end'))
    for m in msgs: m.pinned = True
    return msgs,errors

CRAFT context is read through the gateway from the root down to the dialog's folder. These are context reads, not code execution: startup still imports CRAFT code. Unreadable notebooks and startup failures are reported to Solve-LP as errors, which it turns into context warnings. The kernel sends the dialog's cells with the CRAFT cells in front, the resolved namespace references, and the notebook's settings. Solve-LP owns the policy: system template, mode instructions, ICL examples, and warnings.

In [ ]:
craft_files = [dict(path='CRAFT.ipynb', content=json.dumps(dict(nbformat=4, nbformat_minor=5, metadata={}, cells=[obj2dict(notes)]))),
               dict(path='broken/CRAFT.ipynb', content='not JSON')]
craft_msgs,craft_errors = _craft_messages(craft_files, 'project/dialog.ipynb')
assert any(m.content=='The mascot is a wombat.' for m in craft_msgs)
assert any('broken/CRAFT.ipynb' in e for e in craft_errors)
assert all(m.pinned for m in craft_msgs)
self_craft,self_errors = _craft_messages(craft_files[:1], 'CRAFT.ipynb')
assert not any(m.content=='The mascot is a wombat.' for m in self_craft)
[m.content for m in craft_msgs]

In [ ]:
async def _lp_get(path):
    "GET a Solve-LP route with the kernel's credentials"
    headers = {'Authorization':f'Bearer {os.environ["AAI_USER_KEY"]}', 'X-SolveIt-Instance-Id':os.environ['INSTANCE_ID']}
    return await AsyncTransport().request('GET', os.environ['SOLVELP_URL'].rstrip('/')+path, headers=headers)

async def _resolve_refs(msgs, icl_refs, eval_exprs=None, get_schemas=None):
    "Values, schemas and errors for the `$`, `!` and `&` references in `msgs` and the ICL preamble"
    vs = get_exprs(msgs) + [v for v in icl_refs.get('exprs', []) if v not in get_exprs(msgs)]
    cmds = get_exprs(msgs, sigil='!') + icl_refs.get('cmds', [])
    cmd_exprs = {c:f'get_ipython().getoutput({c!r}).n' for c in cmds}
    eval_exprs = eval_exprs or get_ipython().eval_exprs
    ns = await eval_exprs(vs=vs + list(cmd_exprs.values())) or {}
    if isinstance(ns, str): raise RuntimeError(f'eval_exprs: {ns}')
    undefined = [v for v in vs if is_nameerr(ns.get(v))]
    ns = {v:ns[v] for v in vs if v in ns and v not in undefined} | {f'!`{c}`':ns[e] for c,e in cmd_exprs.items() if e in ns}
    ns = {k:(dict(__bytes__=base64.b64encode(v).decode()) if isinstance(v, bytes) else v) for k,v in ns.items()}
    texts = L(msgs).flatmap(lambda m: [m.content] if m.msg_type in ('prompt','note') else get_output_mds(m.output))
    ts = L(get_refs(' '.join(texts), '&') or []) + icl_refs.get('tools', [])
    ts = [t for t in ts.unique() if t not in ('py','python','spawn_agent')]
    get_schemas = get_schemas or get_ipython().get_schemas
    schema_res = get_schemas(fs=ts)
    if isawaitable(schema_res): schema_res = await schema_res
    if isinstance(schema_res, str): raise RuntimeError(f'get_schemas: {schema_res}')
    schemas = [o for o in schema_res.values() if not isinstance(o, str)]
    errors = [o for o in schema_res.values() if isinstance(o, str)]
    return ns, schemas, dict(vars=undefined, tools=errors)

async def prepare_prompt_context(data, id, fc, *, inc_craft=True, icl_refs=None, solveit=None, eval_exprs=None, get_schemas=None):
    "Assemble a stored prompt's request body for Solve-LP's Solveit policy; namespace lookups default to the active kernel"
    solveit = solveit or {}
    body = _prompt_body(data['cells'], id)
    body['prompt']['metadata'].pop('skipped', None)
    opts = solveit.get('defaults', {}) | data.get('metadata', {}).get('solveit', {})
    opts |= {k:solveit[k] for k in ('math_mode', 'today') if k in solveit}
    crafts,craft_errors = [],[]
    if inc_craft:
        files = await fc.search(path=str(Path(data['path']).parent), up='CRAFT.ipynb', fields='content')
        crafts,craft_errors = _craft_messages(files['files'][::-1], data['path'])
    craft_errors += [e for e in solveit.get('craft_errors') or [] if e]
    messages = crafts + [Message.from_cell(c) for c in body['cells']] + [Message.from_cell(body['prompt'])]
    visible = [m for m in messages if not m.skipped]
    ns,schemas,errors = await _resolve_refs(visible, icl_refs or {}, eval_exprs, get_schemas)
    dialog = dict(path=data['path'], opts=opts, ns=ns, schemas=schemas, errors=errors | dict(crafts=craft_errors),
        py_version=f'{sys.version_info.major}.{sys.version_info.minor}')
    return dict(cells=obj2dict([m.to_cell() for m in messages[:-1]]), prompt=body['prompt'], dialog=dialog)

`prepare_prompt_context` resolves `$` variable references, `!` shell references, and `&` function schemas in the active IPython namespace, including the references Solve-LP's ICL preamble makes (`icl_refs`, from its `dlg_policy` route). It retains skipped flags for Solve-LP, but does not evaluate references inside skipped cells. The result is the request body for `/dlg_responses`: `cells`, `prompt`, and a `dialog` block with the notebook path, effective settings, values, schemas, and errors. Callback arguments let a server-side caller reuse this preparation.

In [ ]:
prompt_context_value = 'wombat'
def prompt_context_example(x:int):
    "Return twice x."
    return x*2

context_target = Message('Recall $`prompt_context_value`, $`prompt_context_missing_782`, and &`prompt_context_example`.', msg_type=sprompt, id='target', output=prompt_output('Stale answer')).to_cell()
context_data = dict(path='project/dialog.ipynb', metadata=dict(solveit=dict(mode='concise', use_tools=False)), cells=[*snapshot[:4], context_target, snapshot[-1]])
context_solveit = dict(defaults=dict(mode='standard', use_tools=True), today='January 1, 2025', math_mode=2, craft_errors=['Startup CRAFT failure'])
context_refs = dict(exprs=['prompt_context_value'], cmds=[], tools=[])
prepared = await prepare_prompt_context(context_data, 'target', None, inc_craft=False, icl_refs=context_refs, solveit=context_solveit)
d = prepared['dialog']
test_eq(d['opts'], dict(mode='concise', use_tools=False, today='January 1, 2025', math_mode=2))
test_eq(d['ns']['prompt_context_value'], 'wombat')
test_eq(d['errors'], dict(vars=['prompt_context_missing_782'], tools=[], crafts=['Startup CRAFT failure']))
test_eq([s['function']['name'] for s in d['schemas']], ['prompt_context_example'])
wire = json.dumps(prepared)
assert 'Stale answer' not in wire and 'Future note' not in wire
test_eq(prepared['prompt']['outputs'], [])
test_eq([c['id'] for c in prepared['cells']], ['notes', 'code', 'previous', 'skipped'])
d

The same request is rebuilt on each execution, so it sees changed variable values rather than a cached namespace. Shell references run through IPython, preserving its current working directory and variable interpolation.

In [ ]:
prompt_context_value = 'platypus'
shell_target = Message('Read $`prompt_context_value` and !`printf prompt_context_shell`.', msg_type=sprompt, id='target').to_cell()
shell_data = context_data | dict(cells=[*snapshot[:4], shell_target, snapshot[-1]])
fresh_context = await prepare_prompt_context(shell_data, 'target', None, inc_craft=False, solveit=context_solveit)
fresh_ns = fresh_context['dialog']['ns']
test_eq(fresh_ns['prompt_context_value'], 'platypus')
assert 'prompt_context_shell' in fresh_ns['!`printf prompt_context_shell`']
fresh_context['dialog']['ns']

This offline CRAFT example supplies the same nearest-folder-first search results as the gateway. Preparation reverses them, includes readable CRAFTs, and retains both read failures and startup failures as warnings.

In [ ]:
class _CraftSearch:
    async def search(self, **kwargs):
        test_eq(kwargs, dict(path='project', up='CRAFT.ipynb', fields='content'))
        return dict(files=craft_files[::-1])

with_crafts = await prepare_prompt_context(context_data, 'target', _CraftSearch(), solveit=context_solveit)
craft_wire = json.dumps(with_crafts)
assert 'craft-block' in craft_wire and 'The mascot is a wombat.' in craft_wire
test_eq(with_crafts['dialog']['errors']['crafts'], [craft_errors[0], 'Startup CRAFT failure'])
with_crafts['cells'][0]['source']

## Tools

The model calls `py` to run code in the user's kernel. The call runs inside the prompt's own execution through the sandbox in the kernel namespace: the `py` object when one is defined, otherwise safepyrun's `python`. A CRAFT can replace it, for example `py = RunPython(yolo=True)` to lift the network restrictions. The return value is the real Python object. Output is captured while the call runs and formatted for the model together with the return value. A permission denial adds the code to the dialog as a code message for the user to run, and tells the model to stop.

In [ ]:
_py_desc = "Run Python/IPython code in the client's persistent execution environment. Variables and imports persist across calls."
_py_params = dict(type='object', properties=dict(code=dict(type='string', description='Python/IPython code to run')), required=['code'], additionalProperties=False)
py_tool = dict(type='function', name='py', description=_py_desc, parameters=_py_params, strict=True)

_spawn_desc = ("Spawn a subagent to complete a task defined by `prompt`. The subagent sees the dialog history "
    "above the current prompt; its reply returns only to the calling model, not the user. Use sparingly, for "
    "discrete tasks whose full working does not need to stay in context.")
_spawn_params = dict(type='object', properties=dict(prompt=dict(type='string', description='The task for the subagent to complete')), required=['prompt'], additionalProperties=False)
spawn_tool = dict(type='function', name='spawn_agent', description=_spawn_desc, parameters=_spawn_params, strict=True)

In [ ]:
def _tool_text(res):
    "The model-facing text for a `py` return value, keeping any marker type"
    if res is None: return ''
    if isinstance(res, Markdown): return MdStr(res.data)
    return res if isinstance(res, str) else repr(res)

def _tool_result(res, cap, aim_info=None):
    "Format a sandboxed call's return value and captured output for the model"
    if isinstance(res, ToolResponse): return res
    outs = [dict(output_type='stream', name=n, text=t) for n,t in (('stdout',cap.stdout), ('stderr',cap.stderr)) if t]
    outs += [dict(output_type='display_data', data=o.data, metadata=o.metadata) for o in cap.outputs]
    val = _tool_text(res)
    if val: outs.append(dict(output_type='execute_result', data={'text/plain':val}, metadata={}))
    m = Message('', msg_type=scode, output=outs)
    merged = merge_media(m.ai_output, output_parts(m, aim_info))
    if not isinstance(merged, str): return merged
    return type(val)(merged or 'No output')

async def _run_py(code, aim_info=None, after=None):
    "Run one `py` tool call in the sandbox; a denial adds the code after message `after` for the user to run"
    ns = get_ipython().user_ns
    sandbox = ns.get('py') or ns['python']
    try:
        with capture_output() as cap: res = await sandbox(code)
    except PermissionError as e:
        if after: await _add_msg_unsafe(code, id=after, msg_type=scode)
        return StopResponse(f"PermissionError: {e}. Code message has been added to dialog. Ask user to run it.")
    return _tool_result(res, cap, aim_info)

In [ ]:
res = await _run_py('1+1')
test_eq((res, type(res)), ('2', str))
res

Return values keep the marker types that `fastllm` and Solveit recognize: `MdStr` for Markdown output, `Safe` to prevent truncation, and `ToolResponse` for structured parts. Printed output comes before the value.

In [ ]:
def md_f(): return Markdown('**aa**')
def safe_f(): return Safe('<b>ok</b>')
md = await _run_py('md_f()')
test_eq((type(md), md), (MdStr, '**aa**'))
test_eq(type(await _run_py('safe_f()')), Safe)
printed = await _run_py('print("hi"); 3')
assert printed.startswith('hi') and printed.endswith('3'), printed
test_eq(await _run_py('x = 1'), 'No output')
printed

A `PermissionError` from the sandbox returns a `StopResponse`, which ends the model's tool loop:

In [ ]:
res = await _run_py("raise PermissionError('stop!')")
test_eq(type(res), StopResponse)
res

A `py` object in the namespace replaces the default sandbox, so a CRAFT can change the policy for every tool call:

In [ ]:
py = RunPython(yolo=True)
test_eq(await _run_py('py.yolo'), 'True')
del py
await _run_py('python.yolo')

`spawn_agent` answers a task as a separate turn over the same prepared context. Its reply returns to the calling model only. A spawned agent has `py` but cannot spawn agents itself.

In [ ]:
def _dlg_chat(model, tools=None, ns=None):
    "An `AsyncChat` against Solve-LP's dialog route, connected with the kernel's environment"
    return AsyncChat(model, api_name='openai', base_url=os.environ['SOLVELP_URL'].rstrip('/'), endpoint='/dlg_responses',
        api_key=os.environ['AAI_USER_KEY'], tools=tools, ns=ns, use_previous_response_id=True,
        extra_headers={'X-SolveIt-Instance-Id':os.environ['INSTANCE_ID']}, prompt_cache_key=os.environ.get('RUSTYGATE_KERNEL_ID'))

_spawn_pre = "<system>You are a spawned agent. Your reply goes only to the calling model, never the user, so answer the task fully and directly.</system>\n"

async def _run_spawn(model, body, py, prompt, think=None):
    "Answer `prompt` as a spawned agent over the prepared turn `body`"
    chat = _dlg_chat(model, [py_tool], dict(py=py))
    pcell = Message(_spawn_pre+prompt, msg_type=sprompt, id='spawn').to_cell()  # a fixed id: the cell exists only inside this one request
    sbody = body | dict(cells=[*body['cells'], body['prompt']], prompt=pcell)
    rs = await chat(stream=True, max_steps=40, think=think, parallel_tool_calls=True, initial_body=sbody)
    async for o in rs: pass
    return chat.full()

## Errors and interruption

A failed or interrupted turn keeps the text already streamed and appends a note. Provider problems get a retry suggestion. Other errors show their message and ask the user to report it.

In [ ]:
def _err_info(e):
    if getattr(e, 'message', None): return e.message
    if getattr(e, 'args', None): return e.args[0]
    return 'unknown error'

def public_error_msg(e):
    "User-facing text for a failed model request"
    info = _err_info(e)
    sc = getattr(e, 'status_code', 0) or 0
    if getattr(e, 'retryable', False) or sc in (408, 409, 429) or sc >= 500:
        return "The currently selected model is experiencing issues. If this keeps happening, please try again later, and try a different model in the meantime."
    if 'media_type' in str(info).lower(): return "The current model doesn't support that media type."
    return f"Whoops! An error ({info}) occurred while processing your request.\nIf this problem persists, please contact us on Discord.\nPlease include your dialog url and error info in your message."

def _note(text, note):
    "`text` with `note` appended, or the note alone for an empty reply"
    return f"{text.rstrip()}\n\n{note}" if text.strip() else note

In [ ]:
e_retry = APIError("boom", status_code=500, retryable=True)
e_media = APIError("invalid media_type", status_code=400, retryable=False)
e_gen = APIError("bad request", status_code=400, retryable=False)
test(public_error_msg(e_retry), 'try a different model', operator.contains)
test_eq(public_error_msg(e_media), "The current model doesn't support that media type.")
test(public_error_msg(e_gen), 'bad request', operator.contains)
test_eq(_note('', INTERRUPTED), INTERRUPTED)
_note('Partial reply\n\n', INTERRUPTED)

In [ ]:
async def _prompt_model_info(model):
    catalog = await _lp_get('/models')
    info = next((m for m in catalog['data'] if m['id']==model), None)
    if info is None: raise ValueError(f'Unsupported model: {model!r}')
    mods = set(info['input_modalities'])
    return info | dict(supports_vision='image' in mods, supports_video_input='video' in mods,
        supports_audio_input='audio' in mods, supports_pdf_input='pdf' in mods)

In [ ]:
async def run_prompt(
    prompt=None, # Prompt text; omit when passing a stored cell id
    *,
    model, # Solve-LP model id
    think=None, # Reasoning effort accepted by fastllm
    id=None, # Read this prompt and its history when execution starts
    solveit=None, # Solveit settings for the policy (`defaults`, `math_mode`, `craft_errors`, `today`); requires id
    inc_craft=True, # Include inherited CRAFT context and startup warnings
):
    "Prepare and stream one prompt in the kernel, running its tool calls in place."
    if (prompt is None) == (id is None): raise ValueError('Pass prompt text or a prompt cell id, not both')
    if solveit is not None and id is None: raise ValueError('Solveit policy requires a stored prompt id')
    info = await _prompt_model_info(model)
    if id is None: body = dict(cells=[], prompt=obj2dict(Message(prompt, msg_type=sprompt).to_cell()))
    else:
        fc = cells_client()
        data = await fc.view(fields='*,meta')
        if solveit is None:
            body = _prompt_body(data['cells'], id)
            skipped = body['prompt']['metadata'].pop('skipped', None)
        else:
            skipped = next(c for c in data['cells'] if c['id']==id)['metadata'].get('skipped')
            body = await prepare_prompt_context(data, id, fc, inc_craft=inc_craft, icl_refs=await _lp_get('/dlg_policy'), solveit=solveit)
        if skipped: await fc.apply([dict(op='merge', id=id, metadata=dict(skipped=None))])
    async def py(code): return await _run_py(code, info, after=id)
    async def spawn_agent(prompt): return await _run_spawn(model, body, py, prompt, think)
    chat = _dlg_chat(model, [py_tool, spawn_tool], dict(py=py, spawn_agent=spawn_agent))
    rs = await chat(stream=True, think=think, max_steps=40, parallel_tool_calls=True, initial_body=body)
    acc,meta = StreamAccum(chat),dict(is_ai_res=True)
    handle = display(raw=True, metadata=meta, display_id=True)
    def show(text):  # round-trip stray surrogates so ZMQ's utf-8 packer can't choke on them
        handle.update({'text/markdown':text.encode('utf-16', 'surrogatepass').decode('utf-16')}, raw=True, metadata=meta)
    try:
        async with aclosing(rs):
            async for item in rs:
                if acc(item): show(acc.txt)
        text = chat.full() + chat.use.fmt()
    except (KeyboardInterrupt, asyncio.CancelledError): text = _note(acc.txt, INTERRUPTED)
    except Exception as e: text = _note(acc.txt, public_error_msg(e))
    show(text)

In [ ]:
#| eval: false
await run_prompt('What is one simple way to reverse a list?', model='openai/gpt-5.6-luna')

Solveit executes `await run_prompt(id=..., model=..., think=..., solveit={...})` with the prompt cell's execution binding. The cell is fetched when the queued execution starts. The gateway saves display updates to that cell without changing its source. `dialoghelper.bootstrap` imports the function through `dialoghelper.stdtools` in newly started kernels.